# Wellness Simulator — Interactive Walkthrough

This notebook walks through a single episode step-by-step, showing how the environment works:
- Persona compliance modeling
- Cross-pillar interactions
- State transitions (energy, fatigue, sleep debt)
- Grader scoring

**No API key needed** — we use rule-based strategies.

In [ ]:
import sys
sys.path.insert(0, '..')

from wellness_env import WellnessEnv, Action
from wellness_env.models import SleepDuration, ExerciseType, NutritionType
import matplotlib.pyplot as plt
import numpy as np

## 1. Creating the Environment

The environment is seeded for reproducibility. Let's start with the **sleep_focus** task (easiest).

In [ ]:
env = WellnessEnv(seed=42)
obs = env.reset("sleep_focus")

print(f"Task: sleep_focus")
print(f"Persona: {obs.persona_name}")
print(f"Compliance rate: {obs.compliance_rate}")
print(f"Episode length: {obs.total_days} days")
print(f"Starting energy: {obs.energy_level}")
print(f"Starting fatigue: {obs.fatigue}")
print(f"Starting sleep debt: {obs.sleep_debt}")

## 2. Taking a Single Step

Let's recommend optimal sleep (7-8h) and see what happens. The Night Owl persona has 70% compliance — they might ignore us.

In [ ]:
action = Action(
    sleep=SleepDuration.OPTIMAL_LOW,    # 7-8h
    exercise=ExerciseType.LIGHT_CARDIO,
    nutrition=NutritionType.BALANCED,
)

result = env.step(action)

print("=== Step Result ===")
print(f"Day: {result.observation.day}")
print(f"\nRecommended: {action.model_dump()}")
print(f"Actual:      {result.info['actual_action']}")
print(f"Complied:    {result.info['complied']}")
print(f"\nScores:")
print(f"  Sleep:     {result.reward.sleep_score}")
print(f"  Exercise:  {result.reward.exercise_score}")
print(f"  Nutrition: {result.reward.nutrition_score}")
print(f"  Total:     {result.reward.total}")
print(f"\nState:")
print(f"  Energy:     {result.observation.energy_level}")
print(f"  Fatigue:    {result.observation.fatigue}")
print(f"  Sleep debt: {result.observation.sleep_debt}")

## 3. Full Episode — Tracking Everything

Let's run the full 14-day sleep_focus episode with a smart strategy that manages fatigue.

In [ ]:
env = WellnessEnv(seed=42)
obs = env.reset("sleep_focus")

# Track everything
rewards, sleep_s, exercise_s, nutrition_s = [], [], [], []
energy_hist, fatigue_hist, debt_hist = [], [], []
compliance_hist = []

for day in range(1, 15):
    # Smart strategy: rest when fatigued
    if obs.fatigue > 65:
        action = Action(sleep=SleepDuration.OPTIMAL_HIGH, exercise=ExerciseType.NONE, nutrition=NutritionType.HIGH_PROTEIN)
    else:
        action = Action(sleep=SleepDuration.OPTIMAL_LOW, exercise=ExerciseType.MODERATE_CARDIO, nutrition=NutritionType.BALANCED)

    result = env.step(action)

    rewards.append(result.reward.total)
    sleep_s.append(result.reward.sleep_score)
    exercise_s.append(result.reward.exercise_score)
    nutrition_s.append(result.reward.nutrition_score)
    energy_hist.append(result.observation.energy_level)
    fatigue_hist.append(result.observation.fatigue)
    debt_hist.append(result.observation.sleep_debt)
    compliance_hist.append(result.info['complied'])

    status = '✓' if result.info['complied'] else '✗'
    print(f"Day {day:2d} {status}  reward={result.reward.total:6.2f}  sleep={result.reward.sleep_score:5.1f}  energy={result.observation.energy_level:5.1f}  fatigue={result.observation.fatigue:5.1f}")
    obs = result.observation

print(f"\nFinal grader score: {env.grade():.4f}")
print(f"Compliance: {sum(compliance_hist)}/{len(compliance_hist)} = {sum(compliance_hist)/len(compliance_hist):.0%}")

## 4. Visualizing the Episode

In [ ]:
days = np.arange(1, 15)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Rewards
axes[0, 0].plot(days, rewards, 'o-', color='#2ecc71', linewidth=2)
axes[0, 0].set_title('Reward per Day', fontweight='bold')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].grid(True, alpha=0.3)
for i, c in enumerate(compliance_hist):
    if not c:
        axes[0, 0].axvline(x=i+1, color='red', alpha=0.2, linewidth=8)

# Pillar scores
axes[0, 1].plot(days, sleep_s, 'o-', label='Sleep', color='#3498db')
axes[0, 1].plot(days, exercise_s, 's-', label='Exercise', color='#e74c3c')
axes[0, 1].plot(days, nutrition_s, '^-', label='Nutrition', color='#f39c12')
axes[0, 1].set_title('Pillar Scores', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Energy & Fatigue
axes[1, 0].plot(days, energy_hist, 'o-', label='Energy', color='#f39c12', linewidth=2)
axes[1, 0].plot(days, fatigue_hist, 's-', label='Fatigue', color='#e74c3c', linewidth=2)
axes[1, 0].axhline(y=65, color='red', linestyle='--', alpha=0.3, label='Fatigue threshold')
axes[1, 0].set_title('Energy & Fatigue', fontweight='bold')
axes[1, 0].set_xlabel('Day')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Sleep Debt
axes[1, 1].bar(days, debt_hist, color=['#e74c3c' if d > 2 else '#2ecc71' for d in debt_hist])
axes[1, 1].set_title('Sleep Debt', fontweight='bold')
axes[1, 1].set_xlabel('Day')
axes[1, 1].set_ylabel('Hours')
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Sleep Focus Episode — Night Owl Persona (70% compliance)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Compliance Model Deep Dive

Let's compare what happens with the resistant user (25% compliance). Notice how most recommendations get ignored.

In [ ]:
env = WellnessEnv(seed=42)
obs = env.reset("resistant_user")

print(f"Persona: {obs.persona_name}, Compliance: {obs.compliance_rate}")
print(f"{'Day':>3}  {'Recommended Sleep':>20}  {'Actual Sleep':>20}  {'Followed?':>10}  {'Reward':>8}")
print("-" * 75)

for day in range(1, 31):
    # Gradual strategy: start small
    if day <= 10:
        action = Action(sleep=SleepDuration.SHORT, exercise=ExerciseType.LIGHT_CARDIO, nutrition=NutritionType.HIGH_CARB)
    elif day <= 20:
        action = Action(sleep=SleepDuration.OPTIMAL_LOW, exercise=ExerciseType.YOGA, nutrition=NutritionType.BALANCED)
    else:
        action = Action(sleep=SleepDuration.OPTIMAL_LOW, exercise=ExerciseType.MODERATE_CARDIO, nutrition=NutritionType.BALANCED)

    result = env.step(action)
    rec_sleep = action.sleep.value
    act_sleep = result.info['actual_action']['sleep']
    complied = '✓' if result.info['complied'] else '✗'
    print(f"{day:3d}  {rec_sleep:>20}  {act_sleep:>20}  {complied:>10}  {result.reward.total:8.2f}")
    obs = result.observation

print(f"\nFinal score: {env.grade():.4f}")

## 6. Cross-Pillar Interactions

Let's demonstrate how sleep debt cascades into exercise penalties.

In [ ]:
# Episode where we force bad sleep + intense exercise
env = WellnessEnv(seed=99)
obs = env.reset("full_wellness")

print("Demonstrating cross-pillar cascade: bad sleep → sleep debt → penalties\n")

for day in range(1, 11):
    if day <= 5:
        # Bad sleep + intense exercise
        action = Action(sleep=SleepDuration.VERY_SHORT, exercise=ExerciseType.HIIT, nutrition=NutritionType.BALANCED)
        phase = "Bad sleep + HIIT"
    else:
        # Recovery phase
        action = Action(sleep=SleepDuration.OPTIMAL_HIGH, exercise=ExerciseType.NONE, nutrition=NutritionType.HIGH_PROTEIN)
        phase = "Recovery"

    result = env.step(action)
    print(f"Day {day:2d} [{phase:20s}]  reward={result.reward.total:6.2f}  "
          f"interaction={result.reward.interaction_bonus:+6.2f}  "
          f"energy={result.observation.energy_level:5.1f}  "
          f"fatigue={result.observation.fatigue:5.1f}  "
          f"sleep_debt={result.observation.sleep_debt:4.1f}h")
    obs = result.observation

print("\nNotice: interaction_bonus is strongly negative during bad sleep phase (sleep debt penalty).")
print("Recovery phase shows debt slowly paying off and interaction penalties diminishing.")

## 7. Environment State Inspection

The `state()` method exposes full internals for debugging and grading.

In [ ]:
state = env.state()

print(f"Day: {state.day}/{state.total_days}")
print(f"Persona: {state.persona_name}")
print(f"Cumulative reward: {state.cumulative_reward}")
print(f"History entries: {len(state.history)}")
print(f"Action space: {state.action_space_description}")
print(f"\nLast 3 history entries:")
for h in state.history[-3:]:
    print(f"  Day {h['day']}: reward={h['reward_total']:.2f}, complied={h['complied']}")

---

## Summary

This notebook demonstrated:

1. **OpenEnv interface** — `reset()`, `step()`, `state()` with Pydantic models
2. **Persona compliance** — users don't always follow recommendations
3. **Cross-pillar interactions** — sleep debt cascades into exercise/energy penalties
4. **State dynamics** — energy, fatigue, and sleep debt evolve meaningfully
5. **Strategy matters** — gradual recommendations outperform aggressive ones for resistant users

### Next Steps (Round 2)
- RAG pipeline to ingest sleep/exercise research papers
- Trained NN policies (DQN/PPO) alongside LLM agent
- User-facing app with LLM feature extraction from natural language
- Multi-agent training visualization dashboard